**Task 07: Querying RDF(s)**

In [45]:
!pip install rdflib
import urllib.request
url = 'https://raw.githubusercontent.com/FacultadInformatica-LinkedData/Curso2025-2026/refs/heads/master/Assignment4/course_materials/python/validation.py'
urllib.request.urlretrieve(url, 'validation.py')
github_storage = "https://raw.githubusercontent.com/FacultadInformatica-LinkedData/Curso2025-2026/master/Assignment4/course_materials"

In [46]:
from validation import Report

First let's read the RDF file

In [47]:
from rdflib import Graph, Namespace, Literal
from rdflib.namespace import RDF, RDFS
# Do not change the name of the variables
g = Graph()
g.namespace_manager.bind('ns', Namespace("http://somewhere#"), override=False)
g.parse(github_storage+"/rdf/data06.ttl", format="TTL")
report = Report()

**TASK 7.1a: For all classes, list each classURI. If the class belogs to another class, then list its superclass.**
**Do the exercise in RDFLib returning a list of Tuples: (class, superclass) called "result". If a class does not have a super class, then return None as the superclass**

In [48]:
# TO DO
# Visualize the results
result = []
for c in g.subjects(RDF.type, RDFS.Class):
    # Busca si la clase tiene superclase
    superclasses = list(g.objects(c, RDFS.subClassOf))
    if superclasses:
        for sc in superclasses:
            result.append((c, sc))
    else:
        result.append((c, None))
for r in result:
  print(r)

(rdflib.term.URIRef('http://oeg.fi.upm.es/def/people#Person'), None)
(rdflib.term.URIRef('http://oeg.fi.upm.es/def/people#Animal'), None)
(rdflib.term.URIRef('http://oeg.fi.upm.es/def/people#Professor'), rdflib.term.URIRef('http://oeg.fi.upm.es/def/people#Person'))
(rdflib.term.URIRef('http://oeg.fi.upm.es/def/people#Student'), rdflib.term.URIRef('http://oeg.fi.upm.es/def/people#Person'))
(rdflib.term.URIRef('http://oeg.fi.upm.es/def/people#FullProfessor'), rdflib.term.URIRef('http://oeg.fi.upm.es/def/people#Professor'))
(rdflib.term.URIRef('http://oeg.fi.upm.es/def/people#AssociateProfessor'), rdflib.term.URIRef('http://oeg.fi.upm.es/def/people#Professor'))
(rdflib.term.URIRef('http://oeg.fi.upm.es/def/people#InterimAssociateProfessor'), rdflib.term.URIRef('http://oeg.fi.upm.es/def/people#AssociateProfessor'))


In [49]:
## Validation: Do not remove
report.validate_07_1a(result)

TASK 7.1a OK


**TASK 7.1b: Repeat the same exercise in SPARQL, returning the variables ?c (class) and ?sc (superclass)**

In [50]:
query = """
SELECT ?c ?sc
WHERE {
  ?c rdf:type rdfs:Class .
  OPTIONAL { ?c rdfs:subClassOf ?sc . }
}
"""

for r in g.query(query):
  print(r.c, r.sc)


http://oeg.fi.upm.es/def/people#Person None
http://oeg.fi.upm.es/def/people#Animal None
http://oeg.fi.upm.es/def/people#Professor http://oeg.fi.upm.es/def/people#Person
http://oeg.fi.upm.es/def/people#Student http://oeg.fi.upm.es/def/people#Person
http://oeg.fi.upm.es/def/people#FullProfessor http://oeg.fi.upm.es/def/people#Professor
http://oeg.fi.upm.es/def/people#AssociateProfessor http://oeg.fi.upm.es/def/people#Professor
http://oeg.fi.upm.es/def/people#InterimAssociateProfessor http://oeg.fi.upm.es/def/people#AssociateProfessor


In [51]:
## Validation: Do not remove
report.validate_07_1b(query,g)

TASK 7.1b OK


**TASK 7.2a: List all individuals of "Person" with RDFLib (remember the subClasses). Return the individual URIs in a list called "individuals"**


In [52]:
ns = Namespace("http://oeg.fi.upm.es/def/people#")

# Obtenemos todas las subclases (recursivamente)
def get_subclasses(cls):
    subclasses = set()
    for s in g.subjects(RDFS.subClassOf, cls):
        subclasses.add(s)
        subclasses.update(get_subclasses(s))
    return subclasses

individuals = []
classes = {ns.Person} | get_subclasses(ns.Person)

for cls in classes:
    for ind in g.subjects(RDF.type, cls):
        individuals.append(ind)

# visualize results
for i in individuals:
  print(i)

http://oeg.fi.upm.es/def/people#Raul
http://oeg.fi.upm.es/def/people#Oscar
http://oeg.fi.upm.es/def/people#Asun


In [53]:
# validation. Do not remove
report.validate_07_02a(individuals)

TASK 7.2a OK


**TASK 7.2b: Repeat the same exercise in SPARQL, returning the individual URIs in a variable ?ind**

In [54]:
query = """
SELECT ?ind
WHERE {
  ?ind rdf:type ?class .
  ?class rdfs:subClassOf* <http://oeg.fi.upm.es/def/people#Person> .
}
"""

for r in g.query(query):
    print(r.ind)
# Visualize the results

http://oeg.fi.upm.es/def/people#Asun
http://oeg.fi.upm.es/def/people#Oscar
http://oeg.fi.upm.es/def/people#Raul


In [55]:
## Validation: Do not remove
report.validate_07_02b(g, query)

TASK 7.2b OK


**TASK 7.3:  List the name and type of those who know Rocky (in SPARQL only). Use name and type as variables in the query**

In [56]:
# TO DO
query =  """
PREFIX ns:  <http://oeg.fi.upm.es/def/people#>
PREFIX rdf:  <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT DISTINCT ?name ?type
  WHERE {
    ?ind ns:knows ns:Rocky .
    ?ind rdf:type ?type .
    OPTIONAL { ?ind rdfs:label ?name }
}
"""
# Visualize the results
for r in g.query(query):
  print(r.name, r.type)


Asun http://oeg.fi.upm.es/def/people#FullProfessor
Raul http://oeg.fi.upm.es/def/people#InterimAssociateProfessor
Fantasma http://oeg.fi.upm.es/def/people#Animal


In [57]:
## Validation: Do not remove
report.validate_07_03(g, query)

TASK 7.3 OK


**Task 7.4: List the name of those entities who have a colleague with a dog, or that have a collegue who has a colleague who has a dog (in SPARQL). Return the results in a variable called name**

In [58]:
# TO DO
query =  """
PREFIX ns:   <http://oeg.fi.upm.es/def/people#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT DISTINCT ?name
  WHERE {
  {
    ?person ns:hasColleague ?colleague1 .
    ?colleague1 ns:ownsPet ?pet .
    ?pet a ns:Animal .
  } UNION {
    ?person ns:hasColleague ?colleague1 .
    ?colleague1 ns:hasColleague ?colleague2 .
    ?colleague2 ns:ownsPet ?pet .
    ?pet a ns:Animal .
  }
  ?person rdfs:label ?name .
}
"""

for r in g.query(query):
    print(r.name)
# Visualize the results

Asun
Oscar
Raul


In [59]:
## Validation: Do not remove
report.validate_07_04(g,query)
report.save_report("_Task_07")

TASK 7.4 OK
